# 02 · A network: the synfire chain

Connect groups of spiking neurons into a network that passes a wave of spikes from group to group. Everything runs on the Brian2 simulator; participants with cluster access run the network on the SpiNNaker2 chip in section 3. Builds on notebook 01 (the `lif` neuron, weights and delays).


Work in pairs. **Try** marks exercises for the pair, **Discuss** marks questions we take up together.

The notebook builds on the vendor's tutorials and adds exercises for the workshop. Each section ends with **Go deeper**: where the content comes from and where to read more.

- Vendor tutorials (SpiNNcloud, Apache-2.0): [py-spinnaker2 `tutorials/deep_dive`](https://gitlab.com/spinnaker2/py-spinnaker2/-/tree/v0.8.5/tutorials/deep_dive), referred to as *tutorial 00*, *01*, …
- API documentation: [spinnaker2.gitlab.io/py-spinnaker2](https://spinnaker2.gitlab.io/py-spinnaker2/)
- **Workshop notes:** the folder [`01_knowledge_base`](../01_knowledge_base/) collects background written while preparing this workshop (spiking networks, the hardware, energy), with figures and further links. It is a reading collection for anyone who wants to delve deeper, not a primary source.

Collapsed blocks (▶) hold background for anyone who wants more; the exercises don't need them.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from spinnaker2 import brian2_sim, helpers, snn

plt.rcParams.update({"axes.grid": True, "axes.axisbelow": True, "grid.color": "0.88", "grid.linewidth": 0.6})  # light grey grid in all figures

brian2 = brian2_sim.Brian2Backend()  # sections 1 and 2 run on Brian2, section 3 switches to the chip

### The py-spinnaker2 API at a glance

Notebook 01 used each of these objects once. Here they are together, with their options: the reference for building the chain below and for the exercise in section 2.

| Object | What it is | How it is created |
|---|---|---|
| `snn.Population` | a group of neurons of one model | `snn.Population(size, neuron_model, params, name, record)` |
| `snn.Projection` | the connections from one population to another | `snn.Projection(pre, post, connections)` |
| `snn.Network` | everything that runs together | `snn.Network(name)`, then `net.add(...)` |
| backend | where the network runs | `brian2_sim.Brian2Backend()` or `hardware.SpiNNcloud48NodeBoard()`, then `hw.run(net, steps)` |

A complete program, line by line:

```python
inp = snn.Population(size=2, neuron_model="spike_list",      # an input: sends spikes at given steps
                     params={0: [1, 5], 1: [3]}, name="inp")  # neuron index -> time steps; unlisted neurons stay silent
out = snn.Population(size=1, neuron_model="lif",              # one leaky integrate-and-fire neuron
                     params={"threshold": 10.0, "alpha_decay": 0.9},
                     name="out", record=["spikes", "v"])      # what to record: spikes, v (potential per step)
conns = [[0, 0, 6, 0], [1, 0, 6, 2]]                          # [pre index, post index, weight, delay]
proj = snn.Projection(pre=inp, post=out, connections=conns)
net = snn.Network("example")
net.add(inp, out, proj)                                       # populations and projections
hw.run(net, 20)                                               # 20 time steps; returns nothing
spikes = out.get_spikes()                                     # results are stored in the population
v = out.get_voltages()
```

**Arguments and options**

- `size`: number of neurons. `name`: must be unique within a network.
- `neuron_model`: `"lif"` for the neuron used in the workshop, `"spike_list"` for an input; the other models are listed in the [API docs, neuron models](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/neuron-models.html).
- `params` of `lif`: `threshold` (default 1.0), `alpha_decay` (0.8), `i_offset` (0), `v_init` (0), `v_reset` (0) and `reset` (`"reset_by_subtraction"` or `"reset_to_v_reset"`). A number applies to every neuron; a list with one value per neuron sets them individually; `reset` is the same for all. A misspelled name stops with an error.
- `record`: a list with `"spikes"` and/or `"v"`. Only what is recorded can be read back.
- `connections`: one `[pre, post, weight, delay]` per connection; weights of `lif` are whole numbers from −15 to 15, delays from 0 to 7. Several projections into the same population add up.

**What comes back**

- `pop.get_spikes()`: a dict `{neuron index: list of spike times}`. A neuron without spikes may have an empty list or be missing, so use `pop.get_spikes().get(i, [])`.
- `pop.get_voltages()`: a dict `{neuron index: array of v}`, one value per time step, recorded after a possible reset. Only if `"v"` is in `record`.
- `net.reset()`: clears all results. A new run merges its results into the old ones, so reset before running the same network again.

**Go deeper:** tutorial 00, sections 3 and 4 (and section 6, a larger synfire chain built in the same steps as section 1 below) (the four building blocks), tutorial 01 (populations, neuron models and parameters), tutorial 02 (connections); [API docs, snn](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/snn-api.html).

## 1. A wave of spikes

A [synfire chain](https://en.wikipedia.org/wiki/Synfire_chain) is a chain of neuron groups, each group connected to the next, like the layers of a feed-forward network. A short burst of input to the first group starts a **volley**: a wave of spikes that travels from group to group.

<details><summary>Where synfire chains come from</summary>

Moshe Abeles proposed synfire chains in the 1980s as a way for the cortex to pass on precisely timed spikes. [Diesmann, Gewaltig and Aertsen (1999)](https://doi.org/10.1038/990101) showed in simulations that a volley either settles into a stable, sharp packet or dies out, depending on how many neurons take part and how synchronous they are, which is what the exercises below explore. Synfire chains are also a common example for spiking simulators and neuromorphic hardware, py-spinnaker2 included: the result is easy to check by eye and sensitive to errors in timing.

</details>

Here: 5 groups of 50 `lif` neurons. Every neuron gets 15 connections from random neurons of the previous group, with weight 4 and delay 2. The threshold is 30, so a neuron needs about 8 input spikes arriving close together to fire. The input is a "kick" of 50 neurons that fire once each, around time step 10, with some random jitter.

<details><summary>Why weights are whole numbers from −15 to 15</summary>

Every synapse is stored in the small local memory of a processor core (128 KB, see the hardware slides) as one packed word with fields for target neuron, delay and weight. For `lif`, the weight field holds a signed integer from −15 to 15 and the delay field 3 bits (0…7). Non-integer weights are rounded toward zero and values outside the range are clipped when the network is mapped, without an error; the Brian2 backend does the same. Variants of the model trade these budgets: `lif_no_delay` drops the delay for 8-bit weights, `lif_32bit` has 20-bit weights. Only the ratio of weight to threshold matters, so this chain's weight 4 with threshold 30 behaves like weight 0.4 with threshold 3. Source: tutorial 02, section 2, and tutorial 01, section 2.

</details>

The chain is built in six small steps, in the same order as the larger synfire chain of tutorial 00, section 6: parameters, input, neurons, connections, run, plot. To explore, change a value in step 1.1 and run the cells 1.1 to 1.6 again, one after the other (Shift+Enter).

### 1.1 Parameters

In [ ]:
# --- change these ---
weight = 4  # of every connection, whole numbers from -15 to 15
threshold = 30
alpha_decay = 0.8  # leak per step: 1 = no leak, smaller = stronger leak
kick_spread = 2  # how far the kick's spike times scatter around step 10 (standard deviation, in steps)

# --- fixed ---
n = 50  # neurons per group
fan_in = 15  # inputs per neuron
delay = 2  # of every connection
n_groups = 5
steps = 40
rng = np.random.default_rng(seed=1)  # fixed seed: same network and input on every run

### 1.2 The kick: input to group 0

A `spike_list` population with 50 neurons, each firing once around step 10. `kick_spread` sets how synchronous the kick is.

In [ ]:
kick_times = rng.normal(10, kick_spread, n).round().clip(1)  # one spike time per kick neuron
kick = snn.Population(size=n, neuron_model="spike_list", params={i: [int(t)] for i, t in enumerate(kick_times)}, name="kick")

### 1.3 Five groups of `lif` neurons

One population per group, all with the same parameters. `reset_to_v_reset` sets $v$ back to 0 after a spike, so every neuron fires at most once per wave.

In [ ]:
lif_params = {"threshold": threshold, "alpha_decay": alpha_decay, "reset": "reset_to_v_reset", "v_reset": 0.0}
groups = [snn.Population(size=n, neuron_model="lif", params=lif_params, name=f"group_{g}", record=["spikes"]) for g in range(n_groups)]

### 1.4 Connections: each group drives the next

One projection per pair of neighbouring groups: kick → group 0 → group 1 → … → group 4. Every neuron of the receiving group gets `fan_in` connections from randomly chosen neurons of the sending group (`rng.permutation(n)[:fan_in]` picks 15 different indices out of 50).

In [ ]:
net = snn.Network("synfire_chain")
net.add(kick, *groups)

for pre, post in zip([kick] + groups[:-1], groups):  # (kick, group 0), (group 0, group 1), ...
    conns = []
    for j in range(n):  # every neuron j of the receiving group ...
        for i in rng.permutation(n)[:fan_in]:  # ... gets fan_in random senders i
            conns.append([int(i), j, weight, delay])
    net.add(snn.Projection(pre=pre, post=post, connections=conns))

print(f"{len(net.projections)} projections, {sum(p.sparse_weights.nnz for p in net.projections)} connections")

### 1.5 Run and read the spikes

`get_spikes()` returns `{neuron index: spike times}` per group. The cell prints, per group, how many spikes there were and over how many time steps they were spread, and counts the synaptic events: every spike is delivered to each of its outgoing connections.

In [ ]:
brian2.run(net, steps)

spike_times = [[t for ts in g.get_spikes().values() for t in ts] for g in groups]  # all spike times of a group
for g, times in enumerate(spike_times):
    print(f"group {g}: {len(times):4d} spikes, " + (f"steps {min(times)}-{max(times)}" if times else "-"))
n_spikes = n + sum(len(times) for times in spike_times[:-1])  # the kick and every group except the last send spikes on
print(f"synaptic events: {n_spikes * fan_in}")  # each spike reaches fan_in neurons on average

### 1.6 Plot

The upper plot shows every spike, the lower one how many neurons fire in each time step. This cell doesn't need changes.

In [ ]:
def plot_chain(steps):
    """Raster of the kick and all groups, and the number of neurons firing per step."""
    fig, (ax_r, ax_h) = plt.subplots(2, 1, figsize=(10, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    ax_r.plot(kick_times, np.arange(n) - n, "|", color="0.6", ms=3)
    for g, group in enumerate(groups):
        idx, t = helpers.spike_times_dict_to_arrays(group.get_spikes())  # {index: times} -> two flat arrays for plotting
        ax_r.plot(t, np.asarray(idx) + g * n, "|", color="black", ms=3)
    ax_r.set_yticks([(g - 0.5) * n for g in range(n_groups + 1)], ["kick"] + [f"group {g}" for g in range(n_groups)])
    ax_r.set(ylabel="neurons", title="every spike")

    bins = np.arange(steps + 1)
    ax_h.bar(bins, np.bincount(kick_times.astype(int), minlength=steps + 1)[: steps + 1], width=0.8, color="0.75")
    for g, group in enumerate(groups):
        times = [t for ts in group.get_spikes().values() for t in ts]
        counts = np.bincount(times, minlength=steps + 1)[: steps + 1]
        ax_h.bar(bins, counts, width=0.8, color="black")
        if times:
            ax_h.text(bins[counts.argmax()], counts.max() + 2, f"{g}", ha="center", va="bottom", fontsize=8)
    ax_h.set(ylim=(0, n * 1.3), ylabel="spikes\nper step", xlim=(0, steps), xlabel="time step (1 step = 1 ms)")
    ax_h.set_title("neurons firing per time step (grey: kick, numbers: group)", fontsize=10)
    plt.tight_layout()
    plt.show()


plot_chain(steps)

Each group fires 3 steps after the previous one (delay 2, plus 1). The kick is spread over several steps, but the volley gets sharper from group to group: by group 2, all 50 neurons fire in the same time step. The lower plot shows it as bars that get narrower and taller from group to group. Each neuron fires once, as soon as enough of its 15 inputs have arrived together. With too few, too weak or too spread-out inputs, fewer neurons reach the threshold, the next group gets even less input, and the volley dies out.

<details><summary>Why the volley gets sharper</summary>

A neuron fires as soon as enough of its 15 inputs have arrived, and it fires only once per volley (reset to 0, the inputs are used up). Every neuron of the next group samples 15 of the 50 spike times, and the early part of the volley already brings most neurons close to the threshold. The spread of the output is therefore smaller than the spread of the input, and after two or three groups all neurons fire in the same step. With too few or too weak inputs, the same mechanism works the other way: fewer neurons reach the threshold, the next group gets even less input, and the volley dies out.

</details>

<details><summary>How the synaptic events are counted</summary>

Every spike of a neuron is delivered to each of its outgoing connections: one synaptic event per connection. Each group has 50 × 15 = 750 incoming connections from 50 neurons, so a neuron has 15 outgoing connections on average. Events = (spikes of the kick and of groups 0–3) × 15; the spikes of the last group go nowhere. On SpiNNaker2, each event costs the receiving core a memory lookup and an addition, so the event count is a proxy for the work done. Whether that shows up in the power reading depends on the size: on our board, 100 active cores are invisible, 1,000 add a few watts (energy demo at the end of the workshop).

</details>

**Try:** change one value in step 1.1, run the cells 1.1 to 1.6 again, and set the value back afterwards. If time is short, do 1 and 2.

1. `weight = 3`: why does the wave die out, although 15 × 3 = 45 is still above the threshold of 30?
2. `kick_spread = 4`: the same input spikes, only less synchronous. What happens?
3. *(if time)* `alpha_decay = 0.5`: a stronger leak. What happens, and how is this related to exercise 2?
4. *(if time)* `threshold = 20`: how often does each neuron fire now, and what happens to the number of synaptic events?

**Discuss:**

- The neurons act as *coincidence detectors*: they fire only if their inputs arrive together. Where is the information in this network: in which neurons fire, or in when they fire?
- On SpiNNaker2, the work, and so the energy, grows with the number of synaptic events, not with the number of neurons (exercise 4). What does that mean for designing networks for this chip?

**Go deeper:** tutorial 00, section 6 is the vendor's showcase version of this network: a larger synfire chain (1,250 neurons, 100,000 synapses) with the `lif_curr_exp` model, local inhibition and an animation of the wave. Tutorial 02, section 3 shows other wiring patterns (one-to-one, all-to-all, random, fixed fan-in), section 4 turns a weight matrix into a connection list, section 5 combines projections. Workshop notes: [why spikes can save energy](../01_knowledge_base/01_primer_snn_basics.md#9-why-this-can-save-energy) and [how spikes travel as packets on the chip](../01_knowledge_base/02_hardware_energy.md#4-communication-spikes-as-packets).

## 2. Build it yourself: close the loop

So far every connection was written for you. Now add one projection yourself: connect the last group back to the first, with the same `fan_in`, `weight` and `delay` as the other connections. Predict first: what happens to the wave after it reaches group 4?

Run the cells 1.1 to 1.5 with the starting values first, so that `net` holds the default chain. Then complete the cell below and run it. It runs 60 steps instead of 40, so a second round of the wave has time to show up.

<details><summary>Solution</summary>

```python
conns = []
for j in range(n):
    for i in rng.permutation(n)[:fan_in]:
        conns.append([int(i), j, weight, delay])
loop = snn.Projection(pre=groups[-1], post=groups[0], connections=conns)
net.add(loop)
```

Group 4 fires all 50 neurons in one step, so every neuron of group 0 gets 15 × 4 = 60 > 30 within one step and fires again: the wave circles through the chain every 15 steps until the run ends. A loop of neurons that keeps itself active is a simple form of memory; with inhibition or weaker weights it would die out.

</details>

In [ ]:
# --- your code: a projection from groups[-1] back to groups[0], added to net ---



# --- run and plot again (doesn't need changes) ---
net.reset()  # clear the results of the previous run
brian2.run(net, 60)
for g, group in enumerate(groups):
    print(f"group {g}: spikes at steps {sorted({t for ts in group.get_spikes().values() for t in ts})}")
plot_chain(60)

## 3. Run the network on the SpiNNaker2 chip (cluster participants)

Set `USE_HARDWARE = True` below and run the cell. It runs the network you built last (in section 1, or with the loop of section 2) on Brian2 and on one 48-chip board and compares the spikes. To see the default chain on the chip, run the cells 1.1 to 1.5 with the starting values first. Run the cell once per pair, early in the hands-on. A run reserves one of the 6 boards and takes about one minute, mostly for powering up and booting the board; if all boards are busy, it waits until one is free. The output reports how many cores the network needs: 6 of the 7,104 a board offers.

Participants without cluster access: read along, the expected result is shown in the workshop.

**Go deeper:** choosing a board and how a network is placed on the cores: tutorial 04. The chip itself: tutorial 06. Workshop notes: [the chip at a glance](../01_knowledge_base/02_hardware_energy.md#1-the-chip-at-a-glance).

In [ ]:
USE_HARDWARE = False  # set to True on the cluster

if USE_HARDWARE:
    from spinnaker2 import hardware

    hw = hardware.SpiNNcloud48NodeBoard()
else:
    hw = brian2_sim.Brian2Backend()

print("Running on backend:", type(hw).__name__)


def all_spikes():
    return [{i: list(ts) for i, ts in g.get_spikes().items() if len(ts)} for g in groups]


net.reset()  # clear earlier results: new results are merged into the old ones
brian2.run(net, steps)
reference = all_spikes()

net.reset()
hw.run(net, steps)
result = all_spikes()

for g, spikes in enumerate(result):
    print(f"group {g}: {sum(len(ts) for ts in spikes.values()):4d} spikes")
print("identical spike times on both backends" if result == reference else "spike times differ")

## 4. What you built

A network of populations and projections, run on Brian2 and on the chip, with the objects of section 0. In notebooks 03 and 04, the NIR import builds exactly these objects automatically from a PyTorch model.

**Go deeper:** the other neuron models (adaptive threshold, Izhikevich, current-based synapses, …) and their weight and delay ranges: tutorial 01, section 2. When a run fails or a network doesn't fit on the chip: tutorial 05. Workshop notes: [other neuron models on the chip](../01_knowledge_base/01_primer_snn_basics.md#5-other-neuron-models-on-the-chip) and [when a network doesn't fit](../01_knowledge_base/02_hardware_energy.md#7-limits-in-practice-when-a-network-doesnt-fit).